# 🎃 ZapalloAI — Notebook 01: Exploración de Datos (EDA)

**Universidad de las Fuerzas Armadas ESPE**

Este notebook analiza los datasets crudos antes de cualquier preprocesamiento:
- Distribución de clases
- Resoluciones y formatos de imagen
- Detección de duplicados
- Visualización de ejemplos por clase
- Estadísticas de calidad (brillo, nitidez)

## ⚠️ Requisito previo
Subir los datasets a Google Drive en `Mi unidad/ZapalloAI/`:
```
ZapalloAI/
├── sweet_pumpkin/   ← Dataset 1 descomprimido (Mendeley)
└── cucurbit_leaf/   ← Dataset 2 descomprimido (Mendeley)
```

In [ ]:
# ── 0. Instalar dependencias ──────────────────────────────────────
!pip install -q imagehash Pillow opencv-python-headless matplotlib seaborn pandas
print('✅ Dependencias instaladas')

In [ ]:
# ── 1. Montar Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ZapalloAI'
SWEET_DIR  = f'{DRIVE_BASE}/sweet_pumpkin'
CUCURB_DIR = f'{DRIVE_BASE}/cucurbit_leaf'

import os
for d in [SWEET_DIR, CUCURB_DIR]:
    exists = os.path.exists(d)
    status = '✅' if exists else '❌ NO ENCONTRADO'
    print(f'{status}: {d}')

In [ ]:
# ── 2. Inventario de clases y conteo de imágenes ──────────────────
from pathlib import Path
import pandas as pd

def inventory_dataset(base_dir: str, dataset_name: str) -> pd.DataFrame:
    """Cuenta imágenes por clase en un dataset."""
    records = []
    base = Path(base_dir)
    if not base.exists():
        print(f'⚠️ {dataset_name}: directorio no encontrado: {base_dir}')
        return pd.DataFrame()
    
    for cls_dir in sorted(base.iterdir()):
        if not cls_dir.is_dir():
            continue
        images = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpeg'))
        records.append({
            'dataset': dataset_name,
            'clase_original': cls_dir.name,
            'n_imagenes': len(images),
            'extensiones': list(set(img.suffix.lower() for img in images))
        })
    return pd.DataFrame(records)

df_sweet  = inventory_dataset(SWEET_DIR,  'Sweet Pumpkin')
df_cucurb = inventory_dataset(CUCURB_DIR, 'Cucurbit Leaf')
df_all = pd.concat([df_sweet, df_cucurb], ignore_index=True)

print('\n📊 Inventario completo de datasets:')
print(df_all.to_string(index=False))
print(f'\n   Total imágenes: {df_all.n_imagenes.sum():,}')

In [ ]:
# ── 3. Distribución de clases — gráfico de barras ────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 120})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2D6A4F', '#52B788', '#74C69D', '#95D5B2', '#D8F3DC']

for ax, (df, name) in zip(axes, [(df_sweet, 'Sweet Pumpkin'), (df_cucurb, 'Cucurbit Leaf')]):
    if df.empty:
        ax.text(0.5, 0.5, f'{name}\nNo encontrado', ha='center', va='center', transform=ax.transAxes)
        continue
    bars = ax.bar(df.clase_original, df.n_imagenes, color=colors[:len(df)])
    ax.set_title(f'{name}\n({df.n_imagenes.sum()} imágenes)', fontweight='bold')
    ax.set_ylabel('Número de imágenes')
    ax.set_xlabel('Clase')
    ax.tick_params(axis='x', rotation=25)
    for bar, val in zip(bars, df.n_imagenes):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
                str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Distribución de Clases por Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/eda_distribucion_clases.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado en Drive')

In [ ]:
# ── 4. Análisis de resoluciones ───────────────────────────────────
from PIL import Image
import numpy as np

def sample_resolutions(base_dir: str, n_sample: int = 50) -> list:
    """Muestra resoluciones de una muestra de imágenes."""
    sizes = []
    base = Path(base_dir)
    if not base.exists():
        return sizes
    all_imgs = list(base.rglob('*.jpg')) + list(base.rglob('*.png'))
    import random
    sample = random.sample(all_imgs, min(n_sample, len(all_imgs)))
    for img_path in sample:
        try:
            with Image.open(img_path) as img:
                sizes.append(img.size)  # (width, height)
        except Exception:
            pass
    return sizes

sweet_sizes  = sample_resolutions(SWEET_DIR)
cucurb_sizes = sample_resolutions(CUCURB_DIR)

print('📐 Resoluciones Sweet Pumpkin:')
if sweet_sizes:
    widths  = [s[0] for s in sweet_sizes]
    heights = [s[1] for s in sweet_sizes]
    print(f'   Ancho:  min={min(widths)}, max={max(widths)}, promedio={np.mean(widths):.0f}')
    print(f'   Alto:   min={min(heights)}, max={max(heights)}, promedio={np.mean(heights):.0f}')

print('\n📐 Resoluciones Cucurbit Leaf:')
if cucurb_sizes:
    widths  = [s[0] for s in cucurb_sizes]
    heights = [s[1] for s in cucurb_sizes]
    print(f'   Ancho:  min={min(widths)}, max={max(widths)}, promedio={np.mean(widths):.0f}')
    print(f'   Alto:   min={min(heights)}, max={max(heights)}, promedio={np.mean(heights):.0f}')

print('\n✅ El modelo espera 224x224 — el redimensionado es automático en YOLO')

In [ ]:
# ── 5. Visualizar 5 ejemplos por clase ────────────────────────────
import random

def show_class_samples(base_dir: str, dataset_name: str, n_per_class: int = 5):
    base = Path(base_dir)
    if not base.exists():
        print(f'⚠️ {dataset_name} no encontrado')
        return
    
    cls_dirs = [d for d in sorted(base.iterdir()) if d.is_dir()]
    n_cls = len(cls_dirs)
    
    fig, axes = plt.subplots(n_cls, n_per_class, figsize=(n_per_class * 3, n_cls * 3))
    fig.suptitle(f'{dataset_name} — Muestras por clase', fontsize=14, fontweight='bold')
    
    if n_cls == 1:
        axes = [axes]
    
    for row_idx, cls_dir in enumerate(cls_dirs):
        images = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
        sample = random.sample(images, min(n_per_class, len(images)))
        
        for col_idx in range(n_per_class):
            ax = axes[row_idx][col_idx]
            if col_idx < len(sample):
                img = Image.open(sample[col_idx]).resize((224, 224))
                ax.imshow(img)
            else:
                ax.axis('off')
            ax.set_xticks([])
            ax.set_yticks([])
            if col_idx == 0:
                ax.set_ylabel(cls_dir.name, fontsize=10, fontweight='bold', rotation=0,
                              labelpad=80, va='center')
    
    plt.tight_layout()
    save_path = f'{DRIVE_BASE}/eda_samples_{dataset_name.lower().replace(" ", "_")}.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'✅ Guardado: {save_path}')

show_class_samples(SWEET_DIR,  'Sweet Pumpkin')
show_class_samples(CUCURB_DIR, 'Cucurbit Leaf')

In [ ]:
# ── 6. Detección de posibles duplicados con perceptual hashing ────
import imagehash
from collections import defaultdict

def find_duplicates(base_dir: str, threshold: int = 5) -> dict:
    """
    Encuentra imágenes duplicadas o casi duplicadas usando pHash.
    threshold: diferencia máxima de hash (0=idénticas, <5=muy similares)
    """
    base = Path(base_dir)
    if not base.exists():
        return {}
    
    hashes = {}
    all_images = list(base.rglob('*.jpg')) + list(base.rglob('*.png'))
    
    print(f'🔍 Analizando {len(all_images)} imágenes...')
    for i, img_path in enumerate(all_images):
        if i % 500 == 0:
            print(f'   {i}/{len(all_images)}...')
        try:
            with Image.open(img_path) as img:
                h = imagehash.phash(img)
            hashes[str(img_path)] = h
        except Exception:
            pass
    
    # Encontrar pares similares
    paths = list(hashes.keys())
    duplicates = defaultdict(list)
    for i in range(len(paths)):
        for j in range(i + 1, len(paths)):
            diff = hashes[paths[i]] - hashes[paths[j]]
            if diff <= threshold:
                duplicates[paths[i]].append((paths[j], diff))
    
    return dict(duplicates)

# ⚠️ Puede tardar varios minutos con datasets grandes
print('🔍 Buscando duplicados en Sweet Pumpkin...')
dups_sweet = find_duplicates(SWEET_DIR, threshold=5)
print(f'   Grupos con duplicados: {len(dups_sweet)}')

print('\n🔍 Buscando duplicados en Cucurbit Leaf...')
dups_cucurb = find_duplicates(CUCURB_DIR, threshold=5)
print(f'   Grupos con duplicados: {len(dups_cucurb)}')

total_dups = len(dups_sweet) + len(dups_cucurb)
if total_dups == 0:
    print('\n✅ No se encontraron duplicados significativos')
else:
    print(f'\n⚠️ Se encontraron {total_dups} grupos con imágenes similares')
    print('   Se eliminarán automáticamente en el Notebook 02')

In [ ]:
# ── 7. Resumen ejecutivo ───────────────────────────────────────────
print('=' * 55)
print('📋 RESUMEN EDA — ZapalloAI')
print('=' * 55)

if not df_all.empty:
    total = df_all.n_imagenes.sum()
    print(f'\nTotal imágenes disponibles: {total:,}')
    print(f'Número de datasets: 2')
    print(f'Número de clases objetivo: 5')
    print(f'\nPor dataset:')
    for ds in ['Sweet Pumpkin', 'Cucurbit Leaf']:
        subset = df_all[df_all.dataset == ds]
        if not subset.empty:
            print(f'  {ds}: {subset.n_imagenes.sum():,} imágenes, {len(subset)} clases')

print(f'\nDuplicados detectados: {total_dups}')
print(f'\n🚀 SIGUIENTE: Ejecutar Notebook 02 para preprocesamiento')
print('=' * 55)